# NB_06 — Pipeline Orchestration

Executes the full data pipeline in the correct dependency order.

Run this notebook to trigger the complete Bronze → Silver → Gold → Data Quality chain.
Each stage runs only if the previous one completed successfully.

Execution order:
1. NB_01 — Bronze Ingestion
2. NB_02 — Silver Transformation
3. NB_03 — Gold Star Schema
4. NB_04 — Data Quality

NB_05 (Insights) is excluded from the automated chain — it is an analytical notebook
intended for on-demand execution by data engineers and analysts.

Author: Diego Santiago Vieira de Brito
Platform: Microsoft Fabric Lakehouse — Synapse PySpark
Last updated: June 2026

In [ ]:
from datetime import datetime

pipeline_start = datetime.now()
print(f"Pipeline started at: {pipeline_start.strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

In [ ]:
# Stage 1 — Bronze Ingestion
# Ingests all source files into Delta tables. No transformations applied.
# Expected output: 12 bronze_ tables in the Lakehouse.

stage = "NB_01_Bronze_Ingestion_v2"
print(f"[{datetime.now().strftime('%H:%M:%S')}] Starting: {stage}")

try:
    result = mssparkutils.notebook.run(stage, timeout_seconds=600)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Completed: {stage}")
except Exception as e:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] FAILED: {stage}")
    print(f"Error: {str(e)}")
    raise RuntimeError(f"Pipeline halted at Bronze Ingestion. Downstream stages will not run.") from e

In [ ]:
# Stage 2 — Silver Transformation
# Cleanses, types, and enriches bronze data.
# Expected output: 12 silver_ tables + bronze_quarantine.

stage = "NB_02_Silver_Transformation_v2"
print(f"[{datetime.now().strftime('%H:%M:%S')}] Starting: {stage}")

try:
    result = mssparkutils.notebook.run(stage, timeout_seconds=900)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Completed: {stage}")
except Exception as e:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] FAILED: {stage}")
    print(f"Error: {str(e)}")
    raise RuntimeError(f"Pipeline halted at Silver Transformation. Gold layer will not be rebuilt.") from e

In [ ]:
# Stage 3 — Gold Star Schema
# Builds the Constellation Star Schema from silver tables.
# Expected output: 3 fact tables, 5 dimension tables, gold_dq_issues shell.

stage = "NB_03_Gold_StarSchema_v2"
print(f"[{datetime.now().strftime('%H:%M:%S')}] Starting: {stage}")

try:
    result = mssparkutils.notebook.run(stage, timeout_seconds=900)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Completed: {stage}")
except Exception as e:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] FAILED: {stage}")
    print(f"Error: {str(e)}")
    raise RuntimeError(f"Pipeline halted at Gold Star Schema. Data Quality checks will not run.") from e

In [ ]:
# Stage 4 — Data Quality
# Runs all 9 DQ checks and writes results to gold_dq_issues.
# Expected output: gold_dq_issues populated. Power BI governance page auto-refreshes.

stage = "NB_04_Data_Quality_v2"
print(f"[{datetime.now().strftime('%H:%M:%S')}] Starting: {stage}")

try:
    result = mssparkutils.notebook.run(stage, timeout_seconds=600)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Completed: {stage}")
except Exception as e:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] FAILED: {stage}")
    print(f"Error: {str(e)}")
    print("Warning: Gold layer is intact. DQ issues table may be stale.")

In [ ]:
# Pipeline summary

pipeline_end = datetime.now()
elapsed = (pipeline_end - pipeline_start).seconds
minutes, seconds = divmod(elapsed, 60)

print("="*60)
print("PIPELINE COMPLETE")
print("="*60)
print(f"Started:  {pipeline_start.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Finished: {pipeline_end.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Elapsed:  {minutes}m {seconds}s")
print()
print("Tables written:")
all_tables = spark.sql("SHOW TABLES").filter(
    "tableName LIKE 'bronze_%' OR tableName LIKE 'silver_%' OR tableName LIKE 'gold_%'"
)
all_tables.select("tableName").orderBy("tableName").show(50, truncate=False)
print(f"Total tables: {all_tables.count()}")